In [1]:
import os
from collections import Counter
import re
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnableLambda

# بارگذاری متغیرهای محیطی
load_dotenv()

# مقداردهی مدل جمینای
# نکته: دما را روی 0.7 تنظیم می‌کنیم تا تنوع فکری ایجاد شود
model = ChatGoogleGenerativeAI(
    model="gemini-flash-lite-latest", 
    temperature=0.7 
)

print("✅ Gemini model initialized with temperature=0.7 for Self-Consistency!")

✅ Gemini model initialized with temperature=0.7 for Self-Consistency!


In [2]:
# طراحی پرامپت CoT
prompt = ChatPromptTemplate.from_template("""
مسئله زیر را حل کن. حتماً مرحله به مرحله فکر کن.
در انتها حتماً یک خط جدید ایجاد کن و فقط با فرمت زیر نتیجه را بنویس:
Final Answer: [عدد نهایی]

مسئله: {question}
""")

# ساخت یک زنجیره واحد که یک بار فکر می‌کند
single_chain = prompt | model | StrOutputParser()

print("✅ Single Base Chain created!")

✅ Single Base Chain created!


In [3]:
# اجرای زنجیره پایه در ۳ شاخه موازی
# هر شاخه به خاطر دمای 0.7 ممکن است مسیر استدلالی متفاوتی را طی کند
consistency_chain = RunnableParallel({
    "path_1": single_chain,
    "path_2": single_chain,
    "path_3": single_chain
})

print("✅ Parallel Execution Chain created (3 Paths)!")

✅ Parallel Execution Chain created (3 Paths)!


In [4]:
def majority_vote(results: dict):
    print("\n--- 🔍 All Generated Paths ---")
    
    extracted_answers = []
    
    # استخراج جواب نهایی از هر مسیر
    for path_name, text in results.items():
        print(f"\n[{path_name}]:\n{text.strip()}")
        
        # پیدا کردن الگوی "Final Answer: X"
        match = re.search(r"Final Answer:\s*(.+)", text, re.IGNORECASE)
        if match:
            extracted_answers.append(match.group(1).strip())
        else:
            extracted_answers.append("UNKNOWN")
            
    print("\n------------------------------")
            
    if not extracted_answers:
        return "هیچ جوابی استخراج نشد!"
        
    # رای‌گیری
    vote_counts = Counter(extracted_answers)
    most_common_answer, count = vote_counts.most_common(1)[0]
    
    return f"\n>>> 🏆 پاسخ نهایی برنده (با {count} رای موافق):\n{most_common_answer}"

# اضافه کردن رای‌گیری به انتهای زنجیره موازی
final_chain = consistency_chain | RunnableLambda(majority_vote)

print("✅ Voting system added to the chain!")

✅ Voting system added to the chain!


In [5]:
question = "I am 6 years old. My sister is half my age. When I am 70, how old will my sister be? Answer in Farsi."

print("⏳ Executing 3 parallel reasoning paths and voting...\n")

# اجرای زنجیره نهایی
output = final_chain.invoke({"question": question})

print(output)

⏳ Executing 3 parallel reasoning paths and voting...



Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.



--- 🔍 All Generated Paths ---

[path_1]:
برای حل این مسئله، مراحل زیر را طی می‌کنیم:

1. **سن فعلی من:** من در حال حاضر ۶ ساله هستم.
2. **سن فعلی خواهرم:** خواهرم نصف سن من است، بنابراین سن او برابر است با:
   $6 \div 2 = 3$ ساله.
3. **محاسبه تفاوت سنی:** اختلاف سنی من و خواهرم همیشه ثابت است. تفاوت سن ما برابر است با:
   $6 - 3 = 3$ سال (خواهرم ۳ سال از من کوچک‌تر است).
4. **سن خواهرم در آینده:** وقتی من ۷۰ ساله شوم، خواهرم همچنان ۳ سال از من کوچک‌تر خواهد بود. بنابراین سن او خواهد بود:
   $70 - 3 = 67$

Final Answer: 67

[path_2]:
برای حل این مسئله، مراحل زیر را طی می‌کنیم:

1. سن فعلی من ۶ سال است.
2. خواهرم نصف سن من است، بنابراین سن فعلی خواهرم برابر است با: $6 \div 2 = 3$ سال.
3. اختلاف سنی من و خواهرم همیشه ثابت می‌ماند. اختلاف سن ما عبارت است از: $6 - 3 = 3$ سال (یعنی خواهرم همیشه ۳ سال از من کوچکتر است).
4. وقتی من ۷۰ ساله شوم، سن خواهرم ۳ سال کمتر از سن من خواهد بود: $70 - 3 = 67$ سال.

Final Answer: 67

[path_3]:
برای حل این مسئله، مراحل زیر را طی می‌کنیم:

1. **سن فعلی من: